In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("/content/drive/MyDrive/Colab Notebooks/E-Commerce Omnichannel Sales Performance & Cohort Retention/data/raw")
PROCESSED_DIR = Path("/content/drive/MyDrive/Colab Notebooks/E-Commerce Omnichannel Sales Performance & Cohort Retention/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
print("Reading raw datasets...")
df_orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
df_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
df_customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
df_products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
df_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

Reading raw datasets...


In [3]:
# Inspect shapes and data samples
print(f"Orders: {df_orders.shape}")
print(f"Order Items: {df_items.shape}")
print(f"Customers: {df_customers.shape}")
print(f"Products: {df_products.shape}")
print(f"Translations: {df_translation.shape}")

Orders: (99441, 8)
Order Items: (112650, 7)
Customers: (99441, 5)
Products: (32951, 9)
Translations: (71, 2)


In [4]:
# 1. Standardize Timestamps
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in datetime_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

# Audit order statuses and null delivery dates
print("Order status breakdown:")
print(df_orders["order_status"].value_counts())

print("\nMissing values in Orders table:")
print(df_orders[datetime_cols].isnull().sum())

Order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing values in Orders table:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [5]:
# Analyze customer identity and repeat purchase behavior
total_sessions = df_customers["customer_id"].nunique()
unique_humans = df_customers["customer_unique_id"].nunique()

print(f"Total Customer Session IDs (customer_id): {total_sessions:,}")
print(f"Actual Unique Customers (customer_unique_id): {unique_humans:,}")
print(f"Repeat Customer Count: {total_sessions - unique_humans:,}")

Total Customer Session IDs (customer_id): 99,441
Actual Unique Customers (customer_unique_id): 96,096
Repeat Customer Count: 3,345


In [7]:
print(df_products)
print(df_translation.head())

                             product_id              product_category_name  \
0      1e9e8ef04dbcff4541ed26657ea517e5                         perfumaria   
1      3aa071139cb16b67ca9e5dea641aaa2f                              artes   
2      96bd76ec8810374ed1b65e291975717f                      esporte_lazer   
3      cef67bcfe19066a932b7673e239eb23d                              bebes   
4      9dc1a7de274444849c219cff195d0b71              utilidades_domesticas   
...                                 ...                                ...   
32946  a0b7d5a992ccda646f2d34e418fff5a0                   moveis_decoracao   
32947  bf4538d88321d0fd4412a93c974510e6  construcao_ferramentas_iluminacao   
32948  9a7c6041fa9592d9d9ef6cfe62a71f8c                    cama_mesa_banho   
32949  83808703fc0706a22e264b9d75f04a2e             informatica_acessorios   
32950  106392145fca363410d287a815be6de4                    cama_mesa_banho   

       product_name_lenght  product_description_lenght  product

In [8]:
# 2. Translate Product Categories & Handle Missing Values
df_products = df_products.merge(df_translation, on="product_category_name", how="left")
df_products["category_name"] = (
    df_products["product_category_name_english"]
    .fillna("other")
    .str.replace("_", " ")
    .str.title()
)
df_products = df_products[[
    "product_id",
    "category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]]

In [9]:
print(df_orders.isnull().sum())
print(df_items.isnull().sum())
print(df_customers.isnull().sum())
print(df_products.isnull().sum())

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
product_id           0
category_name        0
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64


In [11]:
metric_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in metric_cols:
    df_products[col] = df_products[col].fillna(df_products[col].median())

In [12]:
print(df_products.isnull().sum())

product_id           0
category_name        0
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [13]:
# 3. Clean Customers (Keep unique identifiers and geographical fields)
df_customers = df_customers[[
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]]

In [14]:
# Cell 6: Check financial distributions (Prices and Freight costs)
print("Item Price Summary Statistics:")
print(df_items[["price", "freight_value"]].describe())

# Check for zero or negative values
invalid_prices = (df_items["price"] <= 0).sum()
invalid_freight = (df_items["freight_value"] < 0).sum()
print(f"\nInvalid price records: {invalid_prices}")
print(f"Invalid freight records: {invalid_freight}")

Item Price Summary Statistics:
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000

Invalid price records: 0
Invalid freight records: 0


In [15]:
# 4. Export Clean Parquet Files
print("Serializing curated Parquet tables...")
df_orders.to_parquet(PROCESSED_DIR / "orders.parquet", index=False)
df_items.to_parquet(PROCESSED_DIR / "order_items.parquet", index=False)
df_customers.to_parquet(PROCESSED_DIR / "customers.parquet", index=False)
df_products.to_parquet(PROCESSED_DIR / "products.parquet", index=False)

print("Ingestion complete. Parquet files ready in data/processed/.")

Serializing curated Parquet tables...
Ingestion complete. Parquet files ready in data/processed/.
